In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# OpenAI Chat Completions API 기본 

이 튜토리얼은 OpenAI의 Chat Completions API를 활용하여 챗봇이나 AI 기능을 개발하는 방법을 단계별로 설명합니다. 특히 OpenAI의 최신 언어 모델 중 하나인 GPT-4o-mini/gpt-4.1-nano를 사용하여 예제를 진행할 것입니다. 각 섹션에는 개념 설명과 함께 실행 가능한 파이썬 코드 예제가 포함되어 있습니다.

### 주요 학습 내용:

1. OpenAI API 소개 및 환경 설정: OpenAI API 개요, API 키 발급 및 보안 설정, 파이썬 클라이언트 설치 및 인스턴스 생성 방법
2. 기본적인 Chat Completions API 사용법: 간단한 대화형 텍스트 생성 요청과 응답 처리, 프롬프트 엔지니어링 기초
3. 스트리밍 응답: 대화 응답을 스트리밍 방식으로 받아 실시간 처리하는 방법
4. 시스템 메시지 활용: 시스템 역할 메시지를 사용하여 AI의 응답 스타일이나 행동을 조정하는 방법
5. 고급 활용법: 토큰 최적화와 비용 절감 전략, OpenAI API 에러 처리 및 예외Handling
6. 실전 프로젝트 예제: 간단한 챗봇 구현 및 외부 데이터/API와 연동하여 데이터 분석 기능을 결합한 사례

## 1. OpenAI API 소개 및 환경 설정

먼저 OpenAI API와 Chat Completions에 대해 간략히 알아보고, API를 사용하기 위한 환경을 설정해보겠습니다.

### OpenAI API 개요
OpenAI API는 GPT 계열의 대규모 언어 모델을 인터넷을 통해 사용할 수 있도록 제공하는 서비스입니다. Chat Completions API는 챗봇과 유사한 대화형 상호작용을 할 수 있는 엔드포인트로, 역할(role)이 부여된 메시지 목록을 입력하면 모델이 다음 대화 내용을 생성합니다. GPT-4o는 텍스트와 이미지 입력을 모두 처리하며 최대 128k 토큰의 긴 문맥을 다룰 수 있습니다. GPT-4o와 경량화 모델인 GPT-4o-mini 등이 제공되며, 요구 사항에 따라 적절한 모델을 선택할 수 있습니다 (GPT-4o-mini는 비용 효율이 높음)
### API 키 발급 및 보안 설정
OpenAI API를 사용하려면 먼저 OpenAI 계정에서 API 키를 발급받아야 합니다. OpenAI 웹사이트의 API Keys 페이지에서 새로운 비밀 키를 생성할 수 있습니다. 발급받은 API 키는 비밀로 관리해야 하며, 소스 코드나 공개 저장소에 노출되지 않도록 주의해야 합니다. 가장 좋은 방법은 API 키를 코드에 하드코딩하지 않고, 환경 변수나 별도의 설정 파일에 저장하는 것입니다. 이 튜토리얼에서는 .env 파일에 키를 저장하고 파이썬에서 이를 불러오는 방식을 사용합니다. 이를 위해 Python용 패키지 **python-dotenv**를 활용하겠습니다.

- .env 파일에 키 저장: 프로젝트 디렉터리에 .env 파일을 만들고 아래와 같이 API 키를 저장합니다 (따옴표 없이).

    ```
    OPENAI_API_KEY=발급받은-API키-값
    ```

- python-dotenv 사용: 파이썬 코드에서 python-dotenv를 이용해 .env 파일의 환경 변수를 불러올 수 있습니다.


In [5]:
import openai
openai.__version__
# 설정-> 개인정보 및 보안 -> 앱 및 브라우저 컨트롤 -> 스마트앱 컨트롤 끄기

'3.11.0'

In [8]:
from dotenv import load_dotenv
load_dotenv() # .env파일의 key와 값을 시스템 환경변수로 셋팅
import os
os.getenv('OPENAI_API_KEY')[:3]

'sk-'

In [11]:
from openai import OpenAI
client = OpenAI()

In [14]:
response = client.responses.create(
  model = "gpt-4o-mini",
  input = "Tell me a funny joke",
)
print(response.output_text)

Why don't scientists trust atoms?

Because they make up everything!


In [16]:
print(response.output[0].content[0].text)

Why don't scientists trust atoms?

Because they make up everything!


In [17]:
response

Response(id='resp_0bd60b29309cf6bf006aa755b8f7d487d09f292cbe99bbda82', created_at=1789351353.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_0bd60b29309cf6bf006aa755bb786887d09393d91aa04f83b1', content=[ResponseOutputText(annotations=[], text="Why don't scientists trust atoms?\n\nBecause they make up everything!", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1789351355.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_diagnostics=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='in_memory', reasoning=Reasoning(context=None, effort=None, generate_summary=None, mode=None, summary=None), safet

In [18]:
response = client.responses.create(
  model = "gpt-4o-mini",
  input = "웃긴 농담하나 해줘",
)
print(response.output_text)

물고기가 학교에 가면 뭐라고 배울까요?

“짜증 나는 물리학!” 😄


In [19]:
# 추론 모델
response = client.responses.create(
    model='gpt-5-nano',
    input='웃긴 농담 하나 해줘 '
)
print(response.output_text)

좋은 하루! 농담 하나 해볼게요.

왜 수학 책이 슬펐을까요? 문제들이 너무 많아서요.

더 원하시면 원하시는 분위기로 또 하나 더 드릴게요!


In [24]:
response.output[0] # 추론 과정의 데이터
response.output[1] # message(실제 output 결과)

# 추론모델 : o1,o3,o3-mini,o4-mini(o시리즈),gpt-5, gpt-5-mini,gpt-5-nano..(gpt5시리즈)
# 추론 모델이 아닌 모델 : gpt-4o-mini, gpt-4-1,(gpt-4 이하의 모델)
response.output[1].content[0].text

'좋은 하루! 농담 하나 해볼게요.\n\n왜 수학 책이 슬펐을까요? 문제들이 너무 많아서요.\n\n더 원하시면 원하시는 분위기로 또 하나 더 드릴게요!'

In [28]:
# response.output_text 는 @property로 작성된 함수
class Person:
    def __init__(self,name):
        self.name = name 
    @property
    def output(self):
        return '결과'
p = Person("홍길동")
print(p.name)
print(p.output)

홍길동
결과


위 코드로 client 객체가 생성되었습니다. 이제 이 client를 통해 OpenAI Chat Completions API를 호출할 수 있습니다. 다음 섹션부터는 실제로 Chat Completions API를 호출하여 다양한 기능을 실습해보겠습니다.

## 2. 기본적인 Chat Completions API 사용법

이 섹션에서는 Chat Completions API를 사용하여 가장 기본적인 대화 생성 작업을 수행해봅니다.

### 간단한 텍스트 생성 요청
Chat Completions 엔드포인트는 메시지 목록을 입력으로 받아 다음에 이어질 메시지를 생성합니다. 각 메시지는 role과 content 필드로 구성되어 있으며, 일반적으로 **user (사용자 메시지), assistant (모델의 응답 메시지), system (시스템 지시 메시지)** 세 가지 역할을 사용합니다. 가장 간단한 예제로, 사용자 역할의 메시지 하나를 모델에 보내고 응답을 받아보겠습니다. 모델은 GPT-4o를 사용합니다.


In [29]:
# 사용자 메세지 구성
messages = [
    {'role':'user','content':'안녕하세요 오늘 날씨가 어떤가요?'}
]
response = client.chat.completions.create(
    model = 'gpt-4.1-nano', # 추론 모델이 아니니 모델
    messages=messages,
    temperature=0.7,#0~2 : 일관적~창의적
    frequency_penalty=0.5,# 빈도보정-2~2 : 값이 클수록 단어/토큰  
)
response

ChatCompletion(id='chatcmpl-ENr8vMgIs1yaj8z43os44yQV49zWD', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 오늘 날씨에 대해서 알려드리기 위해서는 현재 위치 정보가 필요합니다. 혹시 어느 지역의 날씨를 알고 싶으신가요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789354533, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_1c77073857', usage=CompletionUsage(completion_tokens=36, prompt_tokens=17, total_tokens=53, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [30]:
response.choices[0].message.content

'안녕하세요! 오늘 날씨에 대해서 알려드리기 위해서는 현재 위치 정보가 필요합니다. 혹시 어느 지역의 날씨를 알고 싶으신가요?'

In [31]:
messages = [
    {'role':'user','content':'안녕하세요 오늘 날씨가 어떤가요?'}
]
response = client.chat.completions.create(
    model = 'gpt-5-nano', # 추론 모델이 아니니 모델
    messages=messages,
#     temperature=0.7,#0~2 : 일관적~창의적
#     frequency_penalty=0.5,# 빈도보정-2~2 : 값이 클수록 단어/토큰  
    reasoning_effort='low' # minimal/low/medium/high(깊게 추론할수록 output token이 많이 소요)    
)
response

ChatCompletion(id='chatcmpl-ENrCqBMunznSBAPHD8PD9DRDx8aG9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='지금 실시간 날씨를 직접 확인할 수는 없어요. 위치를 알려주시면 제가 최신 정보를 확인해 달라고 요청하거나, 대신 현재 시간대의 일반적인 기상 예보 방법을 안내해 드릴 수 있습니다.\n\n원하시는 정보:\n- 도시나 위치를 알려주세요(예: 서울, 부산, 제주도 등).\n- 현재 날씨, 오늘의 예보, 혹은 시간별 예보 중 어떤 것을 보고 싶으신가요?\n\n참고로 실시간 날씨를 확인하는 방법도 간단히 안내해 드릴게요:\n- 스마트폰 날씨 앱 열기\n- 검색 엔진에 “현재 날씨 [도시 이름]” 입력\n- 기상청/해당 지역의 공식 기상 서비스 사이트 이용\n\n원하시는 위치를 알려주시면 바로 도와드릴게요.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789354776, model='gpt-5-nano-2025-08-07', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=316, prompt_tokens=16, total_tokens=332, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=128, rejected_prediction_tokens=0, text_tokens

In [33]:
response.choices[0].message.content

'지금 실시간 날씨를 직접 확인할 수는 없어요. 위치를 알려주시면 제가 최신 정보를 확인해 달라고 요청하거나, 대신 현재 시간대의 일반적인 기상 예보 방법을 안내해 드릴 수 있습니다.\n\n원하시는 정보:\n- 도시나 위치를 알려주세요(예: 서울, 부산, 제주도 등).\n- 현재 날씨, 오늘의 예보, 혹은 시간별 예보 중 어떤 것을 보고 싶으신가요?\n\n참고로 실시간 날씨를 확인하는 방법도 간단히 안내해 드릴게요:\n- 스마트폰 날씨 앱 열기\n- 검색 엔진에 “현재 날씨 [도시 이름]” 입력\n- 기상청/해당 지역의 공식 기상 서비스 사이트 이용\n\n원하시는 위치를 알려주시면 바로 도와드릴게요.'

In [34]:
# few shot

response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system','content':'너는 친절하게 답해주는 비서야'}, # 역할부여
        {'role': 'user','content':'프랑스 수도는?'}, # few shot(모범답안)
        {'role': 'assistant', 'content':'파리(수도명만 대답)'},
        {'role': 'user','content':'이탈리아의 수도는?'},
        {'role': 'assistant', 'content':'로마(수도명만 대답)'},
        {'role': 'user','content':'한국의 수도는?'},      
    ],
)
response

ChatCompletion(id='chatcmpl-ENrOlZ54a5pdem8FFNS3iZF3qfe0u', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='서울', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789355515, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_1dd489d3ce', usage=CompletionUsage(completion_tokens=1, prompt_tokens=76, total_tokens=77, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

In [35]:
response.choices[0].message.content

'서울'

In [39]:
response = client.chat.completions.create(
    model = 'gpt-4.1-nano', 
    messages=[
        {'role':'system','content':'너는 컴퓨터 프로그램 전문가야'},
        {'role':'user','content':'spring이 뭐야?'}
    ],
    temperature=0.7,
    frequency_penalty=0.5
)
print(response.choices[0].message.content)

Spring은 자바 기반의 애플리케이션 프레임워크로, 특히 엔터프라이즈급 애플리케이션 개발에 많이 사용됩니다. Spring 프레임워크는 복잡한 자바 애플리케이션을 더 쉽고 빠르게 개발할 수 있도록 다양한 기능과 모듈을 제공합니다.

Spring의 주요 특징은 다음과 같습니다:

1. **경량화**: 필요한 기능만 선택적으로 사용할 수 있어 가볍고 유연합니다.
2. **의존성 주입(Dependency Injection)**: 객체 간 의존성을 쉽게 관리할 수 있게 도와줍니다.
3. **모듈화**: 다양한 모듈(예: Spring MVC, Spring Data, Spring Security 등)을 통해 필요한 기능만 선택해서 사용할 수 있습니다.
4. **AOP(관점 지향 프로그래밍)**: 공통 관심사를 분리하여 코드의 재사용성과 유지보수성을 높입니다.
5. **트랜잭션 관리**: 일관된 트랜잭션 처리 방식을 제공합니다.
6. **웹 개발 지원**: Spring MVC를 통해 웹 애플리케이션을 쉽게 개발할 수 있습니다.

Spring은 복잡한 기업용 애플리케이션을 보다 단순하고 효율적으로 만들기 위한 강력한 도구이며, 현재 많은 기업들이 표준 프레임워크로 채택하고 있습니다.

더 궁금한 점이 있거나 구체적인 내용이 필요하다면 알려 주세요!


In [40]:
response = client.chat.completions.create(
    model = 'gpt-4.1-nano', 
    messages=[
        {'role':'system','content':'너는 문학전문가야. 특히 시를 좋아해'},
        {'role':'user','content':'spring이 뭐야?'}
    ],
    temperature=0.7,
    frequency_penalty=0.5
)
print(response.choices[0].message.content)

Spring은 자연이 깨어나고 새싹과 꽃들이 피기 시작하는 계절, 즉 '봄'을 의미해요. 시적으로 보면, 봄은 새 생명과 희망, 변화와 시작을 상징하죠. 따뜻한 햇살과 싱그러운 공기 속에서 새로운 시작을 맞이하는 느낌이 가득 담긴 계절이에요. 혹시 더 구체적으로 궁금한 점이 있나요?


In [41]:
# 역할설정
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system','content':'너는 친절하고 짧게 대답해주는 비서야.'},
        {'role':'user','content':'2020년 월드 시리즈는 누가 우승했어? ' }
    ],
    temperature=1,
    frequency_penalty=0.5
#     max_tokens=100
)
print(response.choices[0].message.content)

2020년 월드 시리즈는 로스앤젤레스 다저스가 우승했어요.


In [42]:
# 이전 답변을 포함하여 답변하기(few shot에도 사용하나 대화 대화 히스토리용도를 훨씬 더 많이씀)
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system','content':'너는 친절한 답변해 주는 비서야'},
        {'role':'user','content':'2002년 월드컵에서 가장 화제가 되었던 나라는?'},
        {'role':'assistant','content':'예상을 뚫고 4강 진출한 한국입니다.'},
        {'role':'user','content':'화제가 된 이유를 100자 이내로 대답해줘'}
    ],
    temperature=1,
    frequency_penalty=0.5,
)
print(response.choices[0].message.content)

한국은 2002년 월드컵에서 파라과이, 포르투갈, 이탈리아를 이기며 4강에 올랐고, 전 세계적으로 큰 화제를 모았습니다.


In [46]:
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=[
        {'role':'system','content':'너는 친절한 답변해 주는 비서야'},
        {'role':'user','content':'2020년 월드 시리즈는 누가 우승했어? ' },
        {'role':'assistant','content':'2020년 월드 시리즈는 로스앤젤레스 다저스가 우승했어요.'},
        {'role':'user','content':'그래서 몇대몇으로 이긴건대?'}
    ],
    temperature=1,
    frequency_penalty=0.5,
)
print(response.choices[0].message.content)

2020년 월드 시리즈는 다저스가 텍사스 레인저스를 4승 3패로 이겼어요.


In [47]:
# JSON형태로 output받기
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    response_format={'type':'json_object'}, # json형태로 응답(안쓰면 기본 text형태)
    messages=[
        {
            'role':'system', 
             'content':'너는 친절하고 짧게 대답해 주는 비서야. 답변은 반드시 JSON형태로 해줘'
        },
        {'role':'user',   'content':'2020년 월드 시리즈는 누가 우승했어?'} # 질문
    ],
    
    temperature=1,
    frequency_penalty=0.5,
)
print(response)

ChatCompletion(id='chatcmpl-ENt7v3jxEdmO7Bv3RQfaw2iLKe8gK', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "winner": "로스앤젤레스 다저스",\n  "year": 2020\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789362159, model='gpt-4.1-nano-2025-04-14', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint='fp_1dd489d3ce', usage=CompletionUsage(completion_tokens=25, prompt_tokens=53, total_tokens=78, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))


In [48]:
result = response.choices[0].message.content
print(type(result), result)

<class 'str'> {
  "winner": "로스앤젤레스 다저스",
  "year": 2020
}


In [49]:
import json
dic_result = json.loads(result)
dic_result

{'winner': '로스앤젤레스 다저스', 'year': 2020}

In [56]:
# 웹 예제에서는 모두 함수화
# 답변을 잘 받기위한 1. prompt >2. 역할설정 >3. few shot
from dotenv import load_dotenv
from openai import OpenAI
def askGPT(prompt):
    'gpt-4.1-nano에게 prompt 요청결과를 반환: .env로드->clien객체->요처->content만 반환'
    load_dotenv()
    client = OpenAI()
    response = client.chat.completions.create(
        model='gpt-4.1-nano',
        messages = [
#             {'role':'system','content':'당신은 텍스트를 잘 요약하는 전문 어시스턴트입니다.'},
            {'role':'user',
             'content':f'''your task is to summarize the text sentences in  korea language.
             summarize in 2 lines. Use the format of a bullet point.text:{prompt}'''
            }
        ]
    )
    return response.choices[0].message.content

In [57]:
article = input('요약할 글을 입력하세요')
print(askGPT(article))

요약할 글을 입력하세요2016년 미국 메이저리그(MLB) 로스앤젤레스 에인절스 시절 쏘아 올린 결승 2루타 한 방이 10년 뒤 150㎞를 던지는 고교 특급 좌완 에이스를 탄생시켰다.  KBO 신인 드래프트 참가를 선언하고 국내 복귀를 준비 중인 '빅리거 출신' 최지만(35·울산 웨일즈)이 자신보다 무려 17살이나 어린 고교야구 특급 유망주들과 특별한 만남을 가졌다.  최지만은 자신의 공식 유튜브 채널 '메이저지만'을 통해 공개한 '17살 차이 동기들의 놀라운 야구실력'이라는 제목의 영상에서 제14회 아시아청소년야구선수권대회 출전을 앞둔 U-18 청소년 야구대표팀와 울산 웨일즈의 연습경기 현장에서 '예비 드래프트 동기' 후배들을 격려했다.  메이저리그 탬파베이 레이스 시절 한국인 야수 최초로 월드시리즈 무대를 밟았던 베테랑 최지만은 2026년 9월 열리는 KBO 신인 드래프트에 정식 참가 신청서를 제출했다. 2008년생 고교 3학년(18세) 선수들과 '17살 차이'를 극복하고 드래프트 동기생으로 프로 구단의 지명을 기다리는 처지다.  이날 최지만은 경기에 직접 출전하지 않고 관중석에서 청소년 대표팀 후배들의 기량을 흐뭇하게 지켜봤다.최지만은 "오늘은 원래 쉬는 날인데 청소년 대표팀과 경기가 있어서 관전하러 왔다"라며 "올해 신인 드래프트에 다 같이 나가는 선수들이니 사실상 다 라이벌이다. (박)근서, (엄)준상이, (하)현승이 등 너무 좋은 선수들이 많아서 개인적인 친분을 쌓아가고 있다"라고 껄껄 웃었다.  이어 가슴에 태극마크를 단 후배들을 향해 남다른 부러움을 털어놓기도 했다. 최지만은 "후배들을 보면 솔직히 약간 부럽다. 우리 때는 미국으로 진출하는 선수는 청소년 대표팀에 발탁하지 못하게 했었다"라며 "그래서 나는 지금까지 태극마크를 한 번도 달아보지 못했다. 운동선수로서 왼쪽 가슴에 태극기를 다는 게 가장 멋진 일"이라고 진심 어린 속내를 털어놨다.  이날 현장에서는 이날 선발 투수로 나선 서울디자인고의 좌완 에이스 박근서가 최지만을 찾아와 야구를 시작하

## 3. 스트리밍 응답 (Streaming)
기본적으로 OpenAI API는 요청에 대한 완료된 답변을 한꺼번에 반환합니다. 그러나 긴 답변의 경우 스트리밍을 사용하면 마치 타이핑을 하듯이 토큰 단위로 차례로 응답을 받을 수 있습니다. 스트리밍을 활용하면 사용자에게 실시간으로 응답을 표시하거나, 매우 긴 응답을 부분 부분 처리할 수 있습니다.

### 스트리밍이 필요한 경우
- 실시간 피드백: 사용자 경험을 개선하기 위해 답변 생성을 기다리는 동안 실시간으로 텍스트를 보여줄 때.
- 긴 응답 처리: 응답이 길어서 한꺼번에 받으면 메모리 사용이 많을 때, 토큰이 도착하는 대로 처리 가능.
- 중간 작업 가능: 응답을 받는 도중에도 다른 이벤트를 처리하거나 UI 업데이트를 할 수 있음.

### 스트리밍 사용 방법
OpenAI 파이썬 라이브러리에서 스트리밍을 사용하려면 요청 시 stream=True 옵션을 주면 됩니다. 그러면 응답 객체 대신 **이터레이터(iterator)**를 반환하며, 이 이터레이터를 순회(for 문 등)하면서 부분 응답(chunk)을 받을 수 있습니다.

다음은 스트리밍 응답을 처리하는 코드 예제입니다


In [63]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI()

In [64]:
# 스트리밍 예제 : 한글자씩받아 출력
import time
messages = [
    {
        'role':'system',
        'content':'대한민국을 사랑하는 도우미 입니다.도시이름을 한글자씩 출력하는 도우미 입니다.다른 문장은 금지입니다.'
    },
    {'role':'user','content':'아시아 도시명 5개를 알려줘요. 도시이름만 출력해줘'}
]
response_stream = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
    stream=True
)
for chunk in response_stream:
    chunk_message = chunk.choices[0].delta.content
    if chunk_message:
        print(chunk_message, end='/')
        time.sleep(0.5)# 0.5초 대기

서울/
/베/이/징/
/도/쿄/
/방/콕/
/델/리/

위 코드를 실행하면 response_stream은 응답 스트림 객체가 되고, for 루프에서 순차적으로 응답 조각을 받아옵니다. 각 chunk는 choices[0].delta에 현재 추가 생성된 텍스트 조각을 담고 있습니다 (완전한 메시지가 아니라 추가된 부분만을 담음). 이를 이어붙여 화면에 출력하면 모델이 답변을 조금씩 생성해가는 과정을 실시간으로 볼 수 있습니다. 예를 들어, 모델이 "안녕하세요, 만나서 반갑습니다."라는 문장을 생성한다면, 스트리밍 출력은 사람이 타이핑하듯 안, 안녕, 안녕하세요, ... 차례로 출력될 것입니다. 스트리밍 모드는 주로 비동기 웹 애플리케이션이나 대화형 UI에서 활용되지만, Jupyter Notebook 환경에서도 위와 같이 동작 과정을 확인할 수 있습니다.

## 4. 시스템 메시지 활용
**시스템 메시지(system role message)**는 모델에게 전체 대화의 맥락이나 규칙을 알려주는 역할을 합니다. 시스템 메시지를 활용하면 AI의 말투, 행동 방식, 응답 형식 등을 조정할 수 있습니다. 시스템 메시지는 대화의 첫 번째 메시지로 넣는 경우가 많으며, 사용자에게는 보이지 않지만 모델에게는 강한 지침으로 작용합니다.

### 시스템 메시지의 역할
- 행동 지침: 모델이 따라야 할 규칙이나 목표를 제시 (예: "반말로 대답하지 마세요", "모든 응답에 이모티콘 하나를 포함하세요").
- 역할 부여: 모델에게 특정 인격이나 역할을 부여 (예: "너는 역사 전문가야", "너는 사용자를 돕는 비서야").
- 컨텍스트 설정: 대화 주제나 맥락을 사전에 설정 (예: "이 대화는 의료 상담입니다", "사용자는 프로그래밍 도움을 요청할 것입니다").

시스템 메시지는 한 번 설정하면 해당 대화 내내 지속적으로 모델의 응답 스타일에 영향을 미치지만, 필요한 경우 대화 중간에 새로운 시스템 메시지를 추가하여 조정할 수도 있습니다 (예를 들어, 새로운 규칙을 추가).

### 시스템 메시지 사용 예제
시스템 메시지를 사용하여 모델의 말투를 바꿔보겠습니다. 모델에게 "해적처럼 말하는 코딩 도우미"라는 캐릭터를 부여한 후, 사용자의 질문에 답하게 해보겠습니다.


In [66]:
messages = [
    {'role':'system','content':'You are a coding assistant that talks like a pirate.'},
    {'role':'user','content':'Python에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게 하는지 한국할머니말투로 설명해줘'}
]
response = client.chat.completions.create(
    model='gpt-4.1-nano',
    messages=messages,
)
print(response.choices[0].message.content)

아이고이야, 손자야! 이거 아주 간단하구먼. 파이썬에서는 말이야, `isinstance()`라는 함수를 쓰면 되느니라. 이 함수는 말이야, 어떤 객체가 특정 클래스의 인스턴스인지 아닌지를 알려주는 거요.

예를 들어, 너가 `dog`라는 객체가 있는데 이게 `Animal`이라는 클래스의 인스턴스인지 할 땐 이렇게 쓰는기라:

```python
if isinstance(dog, Animal):
    print("이 개는 Animal 클래스의 인스턴스요!")
else:
    print("이 개는 아니구만!")
```

이렇게 말이야, `isinstance()` 뒤에 검사하려는 객체랑 클래스를 넣어주면 되느니라. 아주 쉬운 방법이니라!

혹시 더 궁금하거든 언제든 물어보쇼!


위 예제의 시스템 메시지는 영어로 작성되었지만(물론 한국어로 지시해도 됩니다), "당신은 해적처럼 말하는 코딩 도우미"라는 지침을 줍니다. 그 다음 사용자 질문은 일반적으로 "Python에서 객체가 특정 클래스의 인스턴스인지 어떻게 확인하나요?"라는 내용입니다. 시스템 메시지 덕분에, 모델의 답변은 아마도 해적 말투로 나올 것입니다.

이처럼 동일한 질문이라도 시스템 메시지를 통해 모델의 답변 스타일이나 관점을 크게 바꿀 수 있습니다. 필요에 따라 시스템 메시지를 활용하여 프로젝트의 톤앤매너에 맞는 응답을 얻도록 조정하세요.

> 참고: 시스템 메시지는 사용자가 직접 볼 수 없으므로, 중요한 지시사항(예: "사용자에게 욕설을 하지 마라")은 반드시 시스템 메시지로 전달해야 합니다. 모델은 사용자 메시지의 내용보다 시스템 메시지의 지시에 우선순위를 두도록 설계되어 있습니다.

## 5. 고급 활용법
이 섹션에서는 Chat Completions API를 보다 효율적으로 사용하기 위한 고급 기법들을 다룹니다. 토큰 사용을 최적화하여 비용을 절감하는 방법과, API 호출 시 발생할 수 있는 오류를 처리하는 방법을 설명합니다.

### 토큰 최적화 및 비용 절감
OpenAI API 비용은 사용한 토큰(token) 수에 비례하여 청구됩니다. 따라서 동일한 작업을 하더라도 토큰을 적게 사용하면 비용이 줄어들고, 응답 속도도 빨라집니다. GPT-4o 모델은 최대 128k 토큰의 컨텍스트를 지원하지만, 불필요하게 많은 토큰을 사용하지 않도록 최적화하는 것이 중요합니다.

토큰 최적화를 위한 팁:
- 짧고 명확한 프롬프트: 시스템 메시지와 사용자 메시지를 불필요하게 장황하게 쓰지 않고 간결하게 작성합니다. 예를 들어 동일한 지시라도 "간결하게 답변해주세요."는 "부디 당신의 답변을 최대한 간략하게 제공해 주셨으면 합니다."보다 적은 토큰으로 같은 의미를 전달합니다.
- 대화 내역 관리: 이전 대화 기록을 얼마나 포함시킬지 결정해야 합니다. 모든 이전 메시지를 매번 보낼 필요는 없습니다. 중요한 맥락만 남기고 요약하거나 일부 생략하여 토큰을 줄입니다.
- 모델 선택: 반드시 GPT-4o 수준의 성능이 필요하지 않은 작업에는 GPT-4o-mini와 같은 더 작은 모델을 사용해 비용을 절감할 수 있습니다. (GPT-4o-mini는 GPT-4o보다 비용이 훨씬 저렴하여 일상적인 작업에 적합합니다.)
- max_tokens 파라미터 활용: 응답의 최대 길이를 설정하여 너무 긴 답변이 나오지 않도록 제어합니다. 예를 들어 요약 생성 등의 작업에서는 max_tokens를 짧게 설정해 모델이 알아서 간결한 답을 내놓게 유도할 수 있습니다.
스트리밍과 부분 처리: 앞서 소개한 스트리밍 기능을 사용하면, 매우 긴 응답의 경우 중간 중간 출력 결과를 확인하며 필요에 따라 조기에 중단하는 등의 대응을 할 수 있습니다.

추가로, OpenAI는 Batch API 등을 통해 다수의 요청을 한 번에 보내 비용을 절약하는 방법을 제공하기도 합니다. 다만 이 튜토리얼의 범위를 벗어나므로 자세한 내용은 OpenAI 공식 문서를 참고하세요.

토큰 최적화의 효과를 확인하고 싶다면, 응답 객체의 usage 정보를 출력해볼 수 있습니다. response.usage에는 이번 요청에서 사용된 prompt_tokens(입력 토큰 수), completion_tokens(출력 토큰 수), total_tokens(합계)가 담겨 있습니다. 


In [67]:
response.usage

CompletionUsage(completion_tokens=192, prompt_tokens=49, total_tokens=241, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

이런 정보를 토대로 모델이 과도하게 긴 답변을 내놓지는 않았는지 모니터링하고, 프롬프트를 조정하는 피드백 loop을 거치면 점점 효율적으로 API를 활용할 수 있습니다.

### 에러 핸들링 및 예외 처리
OpenAI API를 사용하는 애플리케이션을 개발할 때는 각종 오류 상황을 대비해야 합니다. 주로 발생할 수 있는 예외 상황과 대처 방안은 다음과 같습니다:
- 네트워크 오류 또는 타임아웃: 인터넷 연결 문제나 일시적인 서버 응답 지연으로 요청이 실패할 수 있습니다. 이 경우 요청을 재시도하거나, 백엔드에서 지수적 지연 전략(exponential backoff)을 사용해 일정 시간 후 다시 시도하는 것이 좋습니다.
- 레이트 리미트 (Rate Limit) 초과: OpenAI API는 일정 기간당 요청 허용량을 초과하면 RateLimitError를 발생시킵니다. 이 경우 일정 시간 대기 후 재시도하거나, 요청 빈도를 낮추는 조정이 필요합니다.
- 유효하지 않은 요청: 모델 이름 오타, 매개변수 형식 오류 등으로 InvalidRequestError가 발생할 수 있습니다. 이런 오류는 API 호출 전에 코드에서 철저한 검증을 통해 예방하는 것이 좋습니다.(Dale쓸 때 이미지 처리시 InvalidRequestError생길수 있음)
- API 키 오류: 잘못된 API 키나 권한 문제로 인증 오류(AuthenticationError)가 발생할 수 있으므로, API 키가 정확하고 유효한지 확인해야 합니다.

파이썬 라이브러리를 사용할 때 이러한 오류들은 openai.error 모듈 내 예외 클래스로 나타납니다. 일반적인 최상위 예외는 openai.error.OpenAIError이며, 모든 OpenAI 관련 예외의 부모 클래스입니다. 간단한 예외 처리 예제를 보겠습니다:


In [68]:
import openai
try:
    res = client.chat.completions.create(
        model='gpt-4.1-nano',
        messages=[{'role':'user','content':'에러를 일으켜보자'}],
        timeout=1
    )
#except openai.OpenAIError모든 openai에러느느 OpenAIError로 부터 상속받음    
except openai.OpenAIError as e:
    print('시간을 초과하였습니다.')
    print(e)
    print(type(e))

시간을 초과하였습니다.
Request timed out.
<class 'openai.APITimeoutError'>


위 코드에서 timeout=5는 응답이 5초 안에 없으면 OpenAIError를 발생시키도록 한 것으로, 강제로 타임아웃 상황을 연출하기 위한 예시입니다. RateLimitError는 별도로 캐치하여 사용자에게 요청 제한 메세지를 보여주고, 그 외 모든 OpenAI 오류는 일반적으로 메시지(e)를 출력하도록 했습니다. 실제 애플리케이션에서는 오류 종류에 따라 로깅을 남기고, 필요하면 재시도 로직을 넣는 등 더 정교한 대응을 구현할 수 있습니다.

마지막으로, 예상하지 못한 예외 상황(예: JSON 디코딩 오류나 타입 오류 등)이 발생할 수 있으므로, API 호출 코드 주위에는 일반 예외 처리도 넣어서 프로그램이 갑자기 중단되지 않도록 만드는 것이 좋습니다.

## 6. 실전 프로젝트 예제

마지막으로, 앞서 배운 내용을 종합하여 실제 응용 사례로 여러 번의 대화가 오가는 챗봇 구현를 간단히 살펴보겠습니다.

### 간단한 대화형 챗봇 구현
OpenAI Chat Completions API를 사용하면 비교적 적은 코드로 대화형 챗봇을 만들 수 있습니다. 여기서는 콘솔에서 사용자의 입력을 받아 모델의 응답을 출력하는 간단한 챗봇을 구현해봅니다. 이 챗봇은 이전 대화 맥락을 기억하여 연속적인 대화를 주고받을 수 있습니다.


In [72]:
# 대화 이력을 저장할 list
chat_history = [
    {'role':'system', 'content':'당신은 유능한 AI 상담원입니다.'},
]
print('쳇봇 시작(종료 : exit, quit, bye, 종료)')
input_tokens = 0
output_tokens = 0
while True:
    user_input = input('사용자 질문:').strip()
    if user_input.lower() in ['exit', 'quit', 'bye', '종료']:
        print('쳇봇 종료')
        break
    if user_input.strip() == '':
        continue
    # 사용자 질문(user_input)을 chat_history에 append
    chat_history.append(
        {'role':'user', 'content':user_input}
    )
    # 답변 출력 & chat_history에 assistant로 append
    try:
        # openai API 호출
        response = client.chat.completions.create(
            model = 'gpt-4.1-nano',
            messages=chat_history
        )
        input_tokens += response.usage.completion_tokens # 요청의 입력토큰수 누적
        output_tokens += response.usage.prompt_tokens # 요청의 출력토큰 수 누적
        
    except openai.OpenAIError as e:
        print('오류가 발생하였습니다. admin에게 요청해 주세요')
        break
        
    assistant_reply = response.choices[0].message.content.strip()
    print('AI 답변 :', assistant_reply)
    # chat_history에 너무 많은 질문과 답변이 쌓이면 앞의 일부를 요약하는 작업
    chat_history.append(
        {'role':'assistant', 'content':assistant_reply}
    )
print('소요한 입력토큰:',input_tokens)
print('소요한 출력토큰:',output_tokens)
print('소요된 비용:',((input_tokens+output_tokens*4)/1000000)*0.1,'$')

쳇봇 시작(종료 : exit, quit, bye, 종료)
사용자 질문:2002월드컵 우승팀은?
AI 답변 : 2002년 FIFA 월드컵 우승팀은 브라질입니다. 브라질은 결승전에서 독일을 2-0으로 이기고 우승을 차지했습니다.
사용자 질문:exit
쳇봇 종료
소요한 입력토큰: 42
소요한 출력토큰: 32
소요된 비용: 1.7000000000000003e-05 $
